In [1]:
import sys 
from bpemb import BPEmb
from tqdm import tqdm
import numpy as np
import os
import torch
from gensim.models import KeyedVectors
from collections import defaultdict

sys.path.append('../datasets')
sys.path.append("..")

## Create BP embedding for MUSE (SINGLE language).

In [2]:
def read(file, threshold=0, vocabulary=None, dtype='float'):
    header = file.readline().split(' ')
    count = int(header[0]) if threshold <= 0 else min(threshold, int(header[0]))
    dim = int(header[1])
    words = []
    matrix = np.empty((count, dim),  dtype=dtype) if vocabulary is None else []
    for i in tqdm(range(count)):
        word, vec = file.readline().split(' ', 1)
        if vocabulary is None:
            words.append(word)
            matrix[i] = np.fromstring(vec, sep=' ',  dtype=dtype)
        elif word in vocabulary:
            words.append(word)
            matrix.append(np.fromstring(vec, sep=' ',  dtype=dtype))
    return (words, matrix) if vocabulary is None else (words, torch.tensor(matrix,  dtype=dtype))

def get_dict(dict_path, source, target):

    dictf = open(dict_path, encoding='utf-8', errors='surrogateescape')
    src2trg = defaultdict(set)

    vocab = set()

    for line in dictf:
        splitted = line.split()
        if len(splitted) > 2:
            # Only using first translation if many are provided
            src, trg = splitted[:2]
        elif len(splitted) == 2:
            src, trg = splitted

        src_ind = source.key_to_index[src]
        trg_ind = target.key_to_index[trg]
        src2trg[src_ind].add(trg_ind)
        vocab.add(src)
    return vocab, src2trg

In [3]:
data_path = '../datasets'

dataset_name = 'muse'
lang = 'en'
emb_dim_source = 100  
emb_dim_target = 50    
vs             = 200000

path_source = os.path.join(data_path, f'muse/embeddings/wiki.multi.{lang}.vec')
path_target = os.path.join(data_path, f'muse/embeddings/wiki.multi.{lang}.vec')

model_source = open(path_source, encoding='utf-8', errors='surrogateescape')
model_target = open(path_target, encoding='utf-8', errors='surrogateescape')

i2w_source, vectors_source = read(model_source)
i2w_target, vectors_target = read(model_target)

bpemb_source = BPEmb(lang=lang, dim=emb_dim_source, vs=vs)
bpemb_target = BPEmb(lang=lang, dim=emb_dim_target, vs=vs)

model_new_source = KeyedVectors(vector_size=emb_dim_source)
model_new_target = KeyedVectors(vector_size=emb_dim_target)

non_valid_indices = set()
valid_indices = set(list(range(len(vectors_source))))

model_new_source.vectors = np.zeros((vectors_source.shape[0], emb_dim_source))
model_new_target.vectors = np.zeros((vectors_target.shape[0], emb_dim_target))

for ix, word in enumerate(tqdm(i2w_source)):  
    word_embed_source = bpemb_source.embed(word)
    word_embed_target = bpemb_target.embed(word)

    if np.isnan(word_embed_source).any() or np.isnan(word_embed_target).any():
        print(word)
        non_valid_indices.add(ix)
        
    if word_embed_source.shape[0] != 1 or word_embed_target.shape[0] != 1:
        non_valid_indices.add(ix)
        continue 

    model_new_source.vectors[ix] = word_embed_source[:]
    model_new_target.vectors[ix] = word_embed_target[:]
    
   
valid_indices = valid_indices - non_valid_indices
print('Number of non valid indices:', len(non_valid_indices))
print('Number of valid indices:', len(valid_indices))

valid_indices = list(valid_indices)

model_new_source.vectors =  model_new_source.vectors[valid_indices]
model_new_target.vectors =  model_new_target.vectors[valid_indices]

model_new_source.index_to_key = [i2w_source[ix] for ix in valid_indices]
model_new_target.index_to_key = [i2w_target[ix] for ix in valid_indices]

model_new_source.key_to_index = {word:i for i, word in enumerate(model_new_source.index_to_key)}
model_new_target.key_to_index = {word:i for i, word in enumerate(model_new_target.index_to_key)}

assert len(model_new_source.vectors) == len(model_new_source.index_to_key)
assert len(model_new_source.vectors) == len(model_new_source.key_to_index.keys())
assert np.isnan(model_new_source.vectors).sum() == 0

assert len(model_new_target.vectors) == len(model_new_target.index_to_key)
assert len(model_new_target.vectors) == len(model_new_target.key_to_index.keys())
assert np.isnan(model_new_target.vectors).sum() == 0
assert len(model_new_source.vectors) == len(model_new_target.vectors)

model_new_source.save(f'../datasets/{dataset_name}_{lang}_BP_{emb_dim_source}_{vs//1000}K.d2v')
model_new_target.save(f'../datasets/{dataset_name}_{lang}_BP_{emb_dim_target}_{vs//1000}K.d2v')

100%|████████████████████████████████████████████████████████████████████████| 200000/200000 [00:16<00:00, 12253.08it/s]


Number of non valid indices: 80832
Number of valid indices: 119168


## Create BP embedding for twitter/wiki-gigaword.

In [3]:
data_path = '../datasets'
dataset_name = 'twitter'

emb_dim_source = 100  
emb_dim_target = 50   
vs             = 200000

path_source = f'../datasets/{dataset_name}_glove_{emb_dim_source}.d2v'
path_target = f'../datasets/{dataset_name}_glove_{emb_dim_target}.d2v'

model_source = KeyedVectors.load(path_source)
model_target = KeyedVectors.load(path_target)

bpemb_source = BPEmb(lang='en', dim=emb_dim_source, vs=vs)
bpemb_target = BPEmb(lang='en', dim=emb_dim_target, vs=vs)

model_new_source = KeyedVectors(vector_size=emb_dim_source)
model_new_target = KeyedVectors(vector_size=emb_dim_target)

model_new_source.vectors = np.zeros((model_source.vectors.shape[0], emb_dim_source))
model_new_target.vectors = np.zeros((model_target.vectors.shape[0], emb_dim_target))

print(model_new_source.vectors.shape)
non_valid_indices = set()
valid_indices = set(list(range(len(model_source.vectors))))

for ix, word in enumerate(tqdm(model_source.index_to_key)):  
    word_embed_source = bpemb_source.embed(word)
    word_embed_target = bpemb_target.embed(word)

    if word_embed_source.shape[0] != 1 or word_embed_target.shape[0] != 1:
        non_valid_indices.add(ix)
        continue
    
    if np.isnan(word_embed_source).any() or np.isnan(word_embed_target).any():
        print(word)
        non_valid_indices.add(ix)

    model_new_source.vectors[ix] = word_embed_source[:]
    model_new_target.vectors[ix] = word_embed_target[:]
    
   
valid_indices = valid_indices - non_valid_indices
print('Number of non valid indices:', len(non_valid_indices))
print('Number of valid indices:', len(valid_indices))

valid_indices = list(valid_indices)

model_new_source.vectors =  model_new_source.vectors[valid_indices]
model_new_target.vectors =  model_new_target.vectors[valid_indices]

model_new_source.index_to_key = [model_source.index_to_key[ix] for ix in valid_indices]
model_new_target.index_to_key = [model_target.index_to_key[ix] for ix in valid_indices]

model_new_source.key_to_index = {word:i for i, word in enumerate(model_new_source.index_to_key)}
model_new_target.key_to_index = {word:i for i, word in enumerate(model_new_target.index_to_key)}

assert len(model_new_source.vectors) == len(model_new_source.index_to_key)
assert len(model_new_source.vectors) == len(model_new_source.key_to_index.keys())
assert np.isnan(model_new_source.vectors).sum() == 0

assert len(model_new_target.vectors) == len(model_new_target.index_to_key)
assert len(model_new_target.vectors) == len(model_new_target.key_to_index.keys())
assert np.isnan(model_new_target.vectors).sum() == 0
assert len(model_new_source.vectors) == len(model_new_target.vectors)

model_new_source.save(f'../datasets/{dataset_name}_BP_{emb_dim_source}_{vs//1000}K.d2v')
model_new_target.save(f'../datasets/{dataset_name}_BP_{emb_dim_target}_{vs//1000}K.d2v')

(1193514, 100)


100%|██████████████████████████████████████████████████████████████████████| 1193514/1193514 [00:48<00:00, 24611.75it/s]


Number of non valid indices: 1101177
Number of valid indices: 92337


## Create embeddings for MUSE (Different languages)

In [3]:
data_path = '../datasets'

source_lang = 'en'
target_lang = 'es'

emb_dim_source  = 100
emb_dim_target  = 100
vs              = 200000

source_path = os.path.join(data_path, f'muse/embeddings/wiki.multi.{source_lang}.vec')
target_path = os.path.join(data_path, f'muse/embeddings/wiki.multi.{target_lang}.vec')
vocab_path  = os.path.join(data_path, f'muse/dictionaries/{source_lang}-{target_lang}.txt') 

source_model = open(source_path, encoding='utf-8', errors='surrogateescape')
target_model = open(target_path, encoding='utf-8', errors='surrogateescape')

i2w_source, vectors_source = read(source_model)
i2w_target, vectors_target = read(target_model)

model_new_source = KeyedVectors(vector_size=emb_dim_source)
model_new_target = KeyedVectors(vector_size=emb_dim_target)

model_new_source.index_to_key = i2w_source[:]
model_new_target.index_to_key = i2w_target[:]

model_new_source.key_to_index = {word: i for i, word in enumerate(i2w_source)}
model_new_target.key_to_index = {word: i for i, word in enumerate(i2w_target)}

100%|████████████████████████████████████████████████████████████████████████| 200000/200000 [00:14<00:00, 13760.86it/s]


In [4]:
vocab, src2trg = get_dict(vocab_path, model_new_source, model_new_target)

valid_keys = []
for k in src2trg.keys():
    if src2trg[k] != set():
        valid_keys.append(k)

print(len(valid_keys))

indices_source = valid_keys[:]
indices_target = [min(src2trg[ix]) for ix in indices_source]

words_source = [model_new_source.index_to_key[ix] for ix in indices_source]
words_target = [model_new_target.index_to_key[ix] for ix in indices_target]

model_new_source.vectors = np.zeros((len(words_source), emb_dim_source))
model_new_target.vectors = np.zeros((len(words_target), emb_dim_target))

model_new_source.index_to_key = words_source
model_new_target.index_to_key = words_target

bpemb_source = BPEmb(lang=source_lang, dim=emb_dim_source, vs=vs)
bpemb_target = BPEmb(lang=target_lang, dim=emb_dim_target, vs=vs)

non_valid_indices = set()
valid_indices = set(list(range(len(model_new_source.vectors))))

for ix in tqdm(range(len(model_new_source.index_to_key))):#(tqdm(i2w_source)):  
    word_source       = model_new_source.index_to_key[ix]
    word_target       = model_new_target.index_to_key[ix]
    
    word_embed_source = bpemb_source.embed(word_source)
    word_embed_target = bpemb_target.embed(word_target)

    if np.isnan(word_embed_source).any() or np.isnan(word_embed_target).any():
        print(word)
        non_valid_indices.add(ix)
        
    if word_embed_source.shape[0] != 1 or word_embed_target.shape[0] != 1:
        non_valid_indices.add(ix)
        continue 

    model_new_source.vectors[ix] = word_embed_source[:]
    model_new_target.vectors[ix] = word_embed_target[:]
    
   
valid_indices = valid_indices - non_valid_indices
print('Number of non valid indices:', len(non_valid_indices))
print('Number of valid indices:', len(valid_indices))

valid_indices = list(valid_indices)

model_new_source.vectors =  model_new_source.vectors[valid_indices]
model_new_target.vectors =  model_new_target.vectors[valid_indices]

model_new_source.index_to_key = [i2w_source[ix] for ix in valid_indices]
model_new_target.index_to_key = [i2w_target[ix] for ix in valid_indices]

#model_new_source.key_to_index = {word:i for i, word in enumerate(model_new_source.index_to_key)}
#model_new_target.key_to_index = {word:i for i, word in enumerate(model_new_target.index_to_key)}

assert len(model_new_source.vectors) == len(model_new_source.index_to_key)
#assert len(model_new_source.vectors) == len(model_new_source.key_to_index.keys())
assert np.isnan(model_new_source.vectors).sum() == 0

assert len(model_new_target.vectors) == len(model_new_target.index_to_key)
#assert len(model_new_target.vectors) == len(model_new_target.key_to_index.keys())
assert np.isnan(model_new_target.vectors).sum() == 0
assert len(model_new_source.vectors) == len(model_new_target.vectors)

#model_new_source.save(f'../datasets/{dataset_name}_{lang}_BP_{emb_dim_source}_{vs//1000}K.d2v')
#model_new_target.save(f'../datasets/{dataset_name}_{lang}_BP_{emb_dim_target}_{vs//1000}K.d2v')

93084


100%|██████████████████████████████████████████████████████████████████████████| 93084/93084 [00:05<00:00, 18023.85it/s]


Number of non valid indices: 31315
Number of valid indices: 61769


In [6]:

model_source_target = KeyedVectors(vector_size=emb_dim_source)
full_len = len(model_new_source.vectors)

if emb_dim_source == emb_dim_target:
    model_source_target.vectors = np.zeros((2*full_len, emb_dim_source), dtype=np.float32)

    for ix in range(2*full_len):
        if ix < full_len:
            model_source_target.vectors[ix] = model_new_source.vectors[ix]
            model_source_target.index_to_key.append(model_new_source.index_to_key[ix])
            
        else:
            model_source_target.vectors[ix] = model_new_target.vectors[ix-full_len]
            model_source_target.index_to_key.append(model_new_target.index_to_key[ix-full_len])
    
    model_source_target.save(f'../datasets/muse_{source_lang}(BP)({emb_dim_source})_{target_lang}(BP)({emb_dim_target}).d2v')

In [27]:

vocab, src2trg = get_dict(vocab_path, source_new, target_new)

valid_keys = []
for k in src2trg.keys():
    if src2trg[k] != set():
        valid_keys.append(k)

print(len(valid_keys))


93084


In [ ]:
vectors_source_new = np.zeros((source_model.vectors.shape[0], source_dim))
vectors_target_new = np.zeros((target_model.vectors.shape[0], target_dim))
words_source = [source_model.i2w[ix] for ix in valid_keys]
words_target = [target_model.i2w[min(src2trg[ix])] for ix in valid_keys]

In [ ]:

bpemb_source = BPEmb(lang=source_lang, dim=source_dim, vs=200000)
bpemb_target = BPEmb(lang=target_lang, dim=target_dim, vs=200000)

In [3]:
!nvidia-smi

Fri Oct 18 16:57:24 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     Off |   00000000:02:00.0 Off |                  N/A |
| 60%   63C    P2            190W /  260W |    8464MiB /  11264MiB |     60%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [30]:
model_test.index_to_key[:10]

['de', '</s>', ',', '.', 'la', 'en', 'el', 'y', '-', ')']

In [89]:
import os
from src.dataset_loaders_diff_emb import embeddings
from src import utils


data_path = '../datasets'

source_lang = 'en'
target_lang = 'es'
        
source_path = os.path.join(data_path, f'muse/embeddings/wiki.multi.{source_lang}.vec.txt')
target_path = os.path.join(data_path, f'muse/embeddings/wiki.multi.{target_lang}.vec.txt')
vocab_path  = os.path.join(data_path, f'muse/dictionaries/{source_lang}-{target_lang}.txt')

source_class = open(source_path, encoding='utf-8', errors='surrogateescape')
target_class = open(target_path, encoding='utf-8', errors='surrogateescape')

i2w_source, vectors_source = read(model)
vectors_source = torch.FloatTensor(vectors_source)
w2i_source = {word: i for i, word in enumerate(i2w_source)}

i2w_target, vectors_target = read(model)
vectors_target = torch.FloatTensor(vectors_target)
w2i_target = {word: i for i, word in enumerate(i2w_target)}

vocab, src2trg = get_dict(vocab_path, source_class, target_class)

valid_keys = []
for k in src2trg.keys():
    if src2trg[k] != set():
        valid_keys.append(k)

words_source = [source_class.i2w[ix] for ix in valid_keys]
words_target = [target_class.i2w[min(src2trg[ix])] for ix in valid_keys]

assert len(random_words_source) == len(random_words_target)

source_class.restrict(random_words_source)
target_class.restrict(random_words_target)

Loading en as source language...


100%|██████████| 200000/200000 [00:20<00:00, 9954.66it/s] 


Loading es as target language...


100%|██████████| 200000/200000 [00:20<00:00, 9955.75it/s] 


In [90]:
from bpemb import BPEmb

bpemb_en_100 = BPEmb(lang="en", dim=100, vs=200000)
bpemb_en_50 = BPEmb(lang="en", dim=50, vs=200000)

In [98]:
from tqdm import tqdm
import numpy as np

source_vectors = np.zeros((source_class.vectors.shape[0], 50))
#target_vectors = np.zeros((target_class.vectors.shape[0], 50))

for ix, word in enumerate(tqdm(source_class.w2i)):  
    #print(word)
    #break
    word_embed_source = np.mean(bpemb_en_50.embed(word), 0, keepdims=True)
    #word_embed_target = np.mean(bpemb_en_50.embed(word), 0, keepdims=True)
    
    source_vectors[ix] = word_embed_source[:]
    #target_vectors[ix] = word_embed_target[:]

100%|██████████| 93084/93084 [00:07<00:00, 12898.13it/s]


In [99]:
source_model_new = KeyedVectors(vector_size=50)
source_model_new.vectors = source_vectors
source_model_new.save(f'../datasets/muse_en_BP_50.d2v')

In [73]:
test_ix = 1902
print(source_class.i2w[test_ix])
print(target_class.i2w[test_ix])

allows
permite


In [11]:
from gensim.models import KeyedVectors

source_target_model = KeyedVectors(vector_size=300)#KeyedVectors.load(f'{data_path}/twitter_{SOURCE_DIM}.d2v')

In [43]:
import numpy as np
from collections import defaultdict

full_len = len(source_class.vectors)

vectors_new = np.zeros((2*full_len, 300), dtype=np.float32)
words_new = []
dict_new = {}
word_count = defaultdict(int)

for ix in range(2*full_len):
    if ix < full_len:
        vectors_new[ix] = source_class.vectors[ix]
        word = source_class.i2w[ix]
        words_new.append(word)
        dict_new[source_class.i2w[ix]] = ix
    else:
        vectors_new[ix] = target_class.vectors[ix-full_len] 
        word = target_class.i2w[ix-full_len]
        words_new.append(word)

        if word in dict_new:
            word_count[word] += 1
            unique_word = f"{word}_{word_count[word]}"  # Create a new unique version
        else:
            word_count[word] = 0
            unique_word = word

        # Add to the dictionary and word list
        dict_new[unique_word] = ix
        #words_new.append(unique_word)

assert len(dict_new.keys()) == 2*full_len
assert len(vectors_new) == 2*full_len
assert len(words_new) == 2*full_len

In [58]:
source_target_model.save('../datasets/muse_og_en_es_300.d2v')

In [87]:
source_target_model = KeyedVectors.load('../datasets/muse_og_en_es_300.d2v')

source_model = KeyedVectors(vector_size=300)
target_model = KeyedVectors(vector_size=300)

full_len = len(source_target_model.vectors)

source_model.vectors = source_target_model.vectors[:full_len//2]
target_model.vectors = source_target_model.vectors[full_len//2:]

source_model.index_to_key = source_target_model.index_to_key[:full_len//2]
target_model.index_to_key = source_target_model.index_to_key[full_len//2:]

for ix, key in enumerate(source_target_model.key_to_index.keys()):
    if ix < full_len//2:
        source_model.key_to_index[key] = ix

    else:
        target_model.key_to_index[key] = ix-full_len//2
    
assert len(target_model.key_to_index.keys()) == full_len//2
assert len(source_model.key_to_index.keys()) == full_len//2

assert len(source_model.vectors) == full_len//2
assert len(target_model.vectors) == full_len//2

assert len(source_model.index_to_key) == full_len//2
assert len(target_model.index_to_key) == full_len//2


In [79]:
print(source_target_model.key_to_index['bin'])
print(source_target_model.index_to_key[4596+full_len//2])
print(source_target_model.index_to_key[4596])

4596
bin
bin


In [66]:
print(len(source_target_model.vectors))
print(len(source_target_model.index_to_key))
print(len(source_target_model.key_to_index.keys()))

186168
186168
186168


In [59]:
test_word = 'hyeueue_je'
test_word[:test_word.find('_')]

'hyeueue'

In [45]:
source_target_model.vectors = vectors_new[:]
source_target_model.index_to_key = words_new[:]
source_target_model.key_to_index = dict_new.copy()

In [57]:
test_ix = 4596
print(source_target_model.key_to_index['bin'])
print(source_target_model.key_to_index['bin_1']-full_len)

4596
4596


In [28]:
print(len(source_class.w2i.keys()))
print(len(source_class.i2w))

93084
93084


In [17]:
print(source_target_model.key_to_index)

{}


In [ ]:
from bpemb import BPEmb

bpemb_en_100 = BPEmb(lang="en", dim=100, vs=200000)
bpemb_en_50 = BPEmb(lang="en", dim=50, vs=200000)

In [150]:
import gensim
from gensim.downloader import load
from gensim.models import KeyedVectors
import sys

SOURCE_DIM = 100
TARGET_DIM = 300

data_path = '../datasets/'
sys.path.append(data_path)

#dataset_name = 'twitter'
dataset_name = 'muse_en_sp'

#source_model = load(dataset_name)

#source_model = KeyedVectors.load(f'{data_path}/{dataset_name}_{SOURCE_DIM}.d2v')
target_model = KeyedVectors.load(f'{data_path}/{dataset_name}_{TARGET_DIM}.d2v')

#source_model_new_emb = KeyedVectors.load(f'{data_path}/{dataset_name}_{SOURCE_DIM}.d2v')
target_model_new_emb = KeyedVectors.load(f'{data_path}/{dataset_name}_{TARGET_DIM}.d2v')

In [151]:
from bpemb import BPEmb

bpemb_en_100 = BPEmb(lang="en", dim=100, vs=200000)
bpemb_en_50 = BPEmb(lang="en", dim=50, vs=200000)

In [6]:
#from tqdm.notebook import tqdm
from tqdm import tqdm
import numpy as np

#source_vectors = np.zeros((source_model.vectors.shape[0], source_model.vectors.shape[1]))
target_vectors = np.zeros((target_model.vectors.shape[0], target_model.vectors.shape[1]))

for ix, word in enumerate(tqdm(target_model.key_to_index)):   
    #word_embed_source = np.mean(bpemb_en_100.embed(word), 0, keepdims=True)
    word_embed_target = np.mean(bpemb_en_25.embed(word), 0, keepdims=True)
    
    #source_vectors[ix] = word_embed_source
    target_vectors[ix] = word_embed_target



  3%|▎         | 35698/1193514 [00:02<01:31, 12632.58it/s]/opt/anaconda3/lib/python3.8/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/anaconda3/lib/python3.8/site-packages/numpy/core/_methods.py:184: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
100%|██████████| 1193514/1193514 [01:33<00:00, 12810.07it/s]


In [7]:
#source_model_new_emb.vectors = source_vectors
target_model_new_emb.vectors = target_vectors
target_model_new_emb.save(f'../datasets/{dataset_name}_BP_{TARGET_DIM}.d2v')

In [8]:
target_model_new_emb.save(f'../datasets/{dataset_name}_BP_{TARGET_DIM}.d2v')

In [131]:
SOURCE_DIM = 100
TARGET_DIM = 50

source_model = KeyedVectors.load(f'{data_path}/{dataset_name}_{SOURCE_DIM}.d2v')
target_model = KeyedVectors.load(f'{data_path}/{dataset_name}_{TARGET_DIM}.d2v')


In [132]:
from tqdm import tqdm
import numpy as np

def read(data_path):
    file = open(data_path, encoding='utf-8', errors='surrogateescape')
    header = file.readline().split(' ')
    count = int(header[0]) #if threshold <= 0 else min(threshold, int(header[0]))
    dim = int(header[1])
    words = []
    matrix = np.empty((count, dim))
    
    for i in tqdm(range(count)):
        word, vec = file.readline().split(' ', 1)
        words.append(word)
        matrix[i] = np.fromstring(vec, sep=' ',  dtype=np.float32)
        
    return words, matrix

data_path_source = '../datasets/muse/embeddings/wiki.multi.en.vec.txt'
data_path_target = '../datasets/muse/embeddings/wiki.multi.es.vec.txt'

words_source, vectors_source = read(data_path_source)
words_target, vectors_target = read(data_path_target)

#vectors_file = open(data_path_vectors, encoding='utf-8', errors='surrogateescape')

100%|██████████| 200000/200000 [00:20<00:00, 9872.33it/s]


In [133]:
w2i_source = {word:ix for ix, word in enumerate(words_source)}
w2i_target = {word:ix for ix, word in enumerate(words_target)}

In [134]:
dictionary = '../datasets/muse/dictionaries/en-es.txt'
vocab = open(dictionary, encoding='utf-8', errors='surrogateescape')

counter = 0
for line in vocab:
    counter+=1
    
print(counter)

112580


In [135]:
vocab = open(dictionary, encoding='utf-8', errors='surrogateescape')

vectors_source_new = np.zeros((counter, 300), dtype=np.float32)
vectors_target_new = np.zeros((counter, 300), dtype=np.float32)

w2i_source_new = {}
w2i_target_new = {}

i2w_source_new = []
i2w_target_new = []

seen_sources = set()
seen_targets = set()

ix = 0

for line in vocab:
    src, trg = line.split()
    
    if src in seen_sources or trg in seen_targets:
        continue
    
    seen_sources.add(src)
    seen_targets.add(trg)

    
    i2w_source_new.append(src)
    i2w_target_new.append(trg)
    
    src_idx = w2i_source[src]
    trg_idx = w2i_target[trg]
    
    vectors_source_new[ix] = vectors_source[src_idx]
    vectors_target_new[ix] = vectors_target[trg_idx]
    
    w2i_source_new[src] = ix
    w2i_target_new[trg] = ix
    #if src=='also' or trg =='igualmente':
    #    print(ix)
    #    print(trg)
    #    print(w2i_target_new[trg])
        
    ix += 1

vectors_source_new = vectors_source_new[:len(i2w_source_new)] 
vectors_target_new = vectors_target_new[:len(i2w_target_new)]
assert len(i2w_source_new)==len(i2w_target_new) and len(vectors_source_new)==len(i2w_target_new) and len(w2i_source_new.keys())==len(i2w_target_new)
print(len(i2w_source_new))
print(len(w2i_source_new.keys()))
print(len(w2i_target_new.keys()))

print( len(vectors_source_new))

print(w2i_target_new['igualmente'])

88296
88296
88296
88296
14


In [136]:
source_model.vectors = vectors_source_new[:]
target_model.vectors = vectors_source_new[:]

source_model.index_to_key = i2w_source_new[:]
target_model.index_to_key = i2w_target_new[:]

source_model.key_to_index = w2i_source_new.copy()
target_model.key_to_index = w2i_target_new.copy()

In [115]:
#source_model.save(f'../datasets/muse_en_300.d2v')
#target_model.save(f'../datasets/muse_es_300.d2v')

In [139]:
print(len(source_model.key_to_index.keys()))
print(len(target_model.key_to_index.keys()))

88296
88296


In [140]:
print(source_model.key_to_index['also'])
print(target_model.key_to_index['igualmente'])

14
14


In [141]:
source_target_model = KeyedVectors.load(f'{data_path}/{dataset_name}_{SOURCE_DIM}.d2v')

In [142]:
source_target_model.vectors = np.concatenate((source_model.vectors, target_model.vectors), axis=0)

aux_list = source_model.index_to_key[:]
aux_list.extend(target_model.index_to_key)
source_target_model.index_to_key = aux_list[:]


aux_dict = source_model.key_to_index.copy()
for key in target_model.key_to_index.keys():
    if key in source_model.key_to_index.keys():
        aux_dict[f'{key}_es'] = target_model.key_to_index[key]
    else:
        aux_dict[key] = target_model.key_to_index[key]
#print(len(aux_dict.keys()))
#aux_dict.update(target_model.key_to_index)
#print(len(target_model.key_to_index.keys()))

print(len(aux_dict.keys()))

source_target_model.key_to_index = aux_dict.copy()

176592


In [146]:
print(len(source_target_model.key_to_index.keys()))

176592


In [144]:
test_ix = 14
full_len = len(source_target_model.vectors)
print(source_target_model.index_to_key[test_ix])
print(source_target_model.index_to_key[full_len//2+test_ix])

print(source_target_model.key_to_index['also'])
print(source_target_model.key_to_index['igualmente'])

also
igualmente
14
14


In [94]:
print(dict(list(source_target_model.key_to_index.items())[0: full_len//2]) )

{'the': 0, 'and': 1, 'was': 2, 'for': 3, 'that': 4, 'with': 5, 'from': 6, 'this': 7, 'utc': 8, 'his': 9, 'not': 10, 'are': 11, 'talk': 12, 'which': 13, 'also': 14, 'has': 15, 'were': 16, 'but': 17, 'have': 18, 'one': 19, 'new': 20, 'first': 21, 'page': 22, 'you': 23, 'they': 24, 'had': 25, 'article': 26, 'who': 27, 'all': 28, 'their': 29, 'there': 30, 'been': 31, 'made': 32, 'people': 33, 'may': 34, 'after': 35, 'other': 36, 'should': 37, 'two': 38, 'score': 39, 'her': 40, 'can': 41, 'more': 42, 'about': 43, 'when': 44, 'time': 45, 'team': 46, 'american': 47, 'such': 48, 'discussion': 49, 'links': 50, 'only': 51, 'some': 52, 'see': 53, 'united': 54, 'years': 55, 'into': 56, 'school': 57, 'world': 58, 'university': 59, 'during': 60, 'out': 61, 'state': 62, 'states': 63, 'national': 64, 'wikipedia': 65, 'year': 66, 'most': 67, 'city': 68, 'over': 69, 'used': 70, 'then': 71, 'than': 72, 'county': 73, 'external': 74, 'where': 75, 'will': 76, 'what': 77, 'delete': 78, 'any': 79, 'these': 80

In [145]:
source_target_model.save(f'../datasets/muse_en_sp_300.d2v')

In [148]:
print(len(list(source_target_model.key_to_index.keys())))


176592


In [147]:
source_target_model = KeyedVectors.load(f'../datasets/muse_en_sp_300.d2v')

In [130]:
dict_1 = {'a_es':0, 'desdd_es':1, 'jeieiek':2}
saved_keys = list(dict_1.keys())
for key in saved_keys:
    if '_es' in key:
        dict_1[key[:key.find('_es')]] = dict_1[key]
        dict_1.pop(key, None)
        
print(dict_1)

{'jeieiek': 2, 'a': 0, 'desdd': 1}


In [39]:
test_ix = 41
print(source_model.index_to_key[test_ix])
print(target_model.index_to_key[test_ix])

her
ella


In [16]:
test_ix = 145
print(i2w_source_new[test_ix])
print(i2w_target_new[test_ix])



same
mismo


In [24]:
print(source_model_new_emb.vectors[357])

[ 0.366923   -0.21438999 -0.47883099 -0.169989   -1.227265    0.26386601
 -0.444143   -0.145785   -0.15522601 -0.60230601 -0.079181   -0.108432
  0.142757   -0.341106    0.49753201  0.127662    0.547001    0.222452
 -0.084726   -0.051751   -0.007804    0.68220103 -0.087677    0.36475199
 -0.074393   -0.03405    -0.329263   -0.053278    0.34968501 -0.31172299
  0.447869    0.151962    0.52823901 -0.105502   -0.28155401 -0.43827999
 -0.255817    0.086996   -0.176595   -0.088176   -0.056154   -0.37741399
  0.50273299  0.011499    0.62182498  0.235295   -0.417299    0.205706
  0.43533501 -0.060655   -0.30161101  0.144713    0.245987    0.02667
 -0.38263199  0.20138     0.69739902 -0.68095899  0.25249201 -0.047652
 -0.33043    -0.101697    0.37259799 -0.440097   -1.19067705  0.34818301
  0.26504901 -0.01703    -0.058476   -0.31902999  0.39280099 -0.53339201
  0.079584   -0.066565   -0.20047601  0.33654499 -0.054512    0.29371101
  0.166618   -0.152321   -0.21380299 -0.066169    0.31286299  

In [16]:
print(source_model_new_emb.vectors[0])
print(source_model.vectors[0])

[ 0.0828514   0.0421692   0.222037    0.56030422  0.98128045  0.10918641
 -0.59110141  0.1357574   0.25139958 -0.1517012  -0.26522523  0.014587
  0.11271419  0.2658146   0.09768461 -0.1070354   0.1120898   0.19834699
 -0.243071    0.05079759  0.29027939  0.25669283 -0.48950881 -0.45913419
  0.1232978   0.010211   -0.12513061 -0.0633032  -0.247697    0.15302899
 -0.43865162  0.182096   -0.47076923  0.11381821  0.0846874  -0.0422
  0.0029582   0.1563044   0.043838    0.3090966   0.27891102 -0.44998717
  0.1852518  -0.07594819 -0.0973314  -0.10483839  0.0681748  -0.0720912
 -0.5318954   0.1028082  -0.3509658  -0.1450786  -0.16422579 -0.29003581
  0.19497499 -0.38782898 -0.068606    0.27785918 -0.0015964  -0.0276414
  0.3384698  -0.16349439 -0.0717182  -0.0100446  -0.47538456  0.13915579
 -0.16386819  0.42251998  0.4892436  -0.2373784   0.39162299  0.0210092
 -0.05334859  0.131906    0.1629546   0.0754976   0.0192556  -0.15720519
 -0.11922201 -0.03359241 -0.0424978   0.2885128   0.0083738 

In [5]:
!pip install nodejs

  Created wheel for nodejs: filename=nodejs-0.1.1-py3-none-any.whl size=3492 sha256=949cc21a9319117e892c8b1acebfa7073ec5fde064b7ba30828e6dac3537c213
  Stored in directory: /root/.cache/pip/wheels/b3/ce/d8/40f8634e964582985b2c4560cbba06d4f50d3da980fccd0497
  Created wheel for optional-django: filename=optional_django-0.1.0-py3-none-any.whl size=9979 sha256=130757b4769520f80adfa7b531b6511a2a13dd7e0f5ea07016d705823018e6e8
  Stored in directory: /root/.cache/pip/wheels/3b/42/9c/10c5c4021a4edf8416f18e88b427b92e99f187e61e15e08100
Successfully built nodejs optional-django


In [81]:
test_ix = 0 
w2i = list(source_model.key_to_index.keys())
a = w2i[test_ix]

print(a)
print(source_model.vectors[test_ix])

<user>
[ 0.63006    0.65177    0.25545    0.018593   0.043094   0.047194
  0.23218    0.11613    0.17371    0.40487    0.022524  -0.076731
 -2.2911     0.094127   0.43293    0.041801   0.063175  -0.64486
 -0.43657    0.024114  -0.082989   0.21686   -0.13462   -0.22336
  0.39436   -2.1724    -0.39544    0.16536    0.39438   -0.35182
 -0.14996    0.10502   -0.45937    0.27729    0.8924    -0.042313
 -0.009345   0.55017    0.095521   0.070504  -1.1781     0.013723
  0.17742    0.74142    0.17716    0.038468  -0.31684    0.08941
  0.20557   -0.34328   -0.64303   -0.878     -0.16293   -0.055925
  0.33898    0.60664   -0.2774     0.33626    0.21603   -0.11051
  0.0058673 -0.64757   -0.068222  -0.77414    0.13911   -0.15851
 -0.61885   -0.10192   -0.47       0.19787    0.42175   -0.18458
  0.080581  -0.22545   -0.065129  -0.15328    0.087726  -0.18817
 -0.08371    0.21779    0.97899    0.1092     0.022705  -0.078234
  0.15595    0.083105  -0.6824     0.57469   -0.19942    0.50566
 -0.18277   

In [7]:
from bpemb import BPEmb
bpemb_en_100 = BPEmb(lang="en", dim=100)
bpemb_en_50 = BPEmb(lang="en", dim=50)

In [77]:

print(bpemb_en_100.encode(a))
print(bpemb_en_50.encode(a))

['▁', '<', 'us', 'er', '>']
['▁', '<', 'us', 'er', '>']


In [64]:
a1 = bpemb_en_100.encode(a)
print(a1)

['▁the']


In [70]:
a2 = bpemb_en_50.most_similar(a)

KeyError: "Key 'dksmfkdf' not present in vocabulary"

In [82]:
print(bpemb_en_100.embed(a).shape)#.vectors[bpemb_en_100.emb.key_to_index[a]])

(5, 100)


In [68]:
print(source_model.vectors[test_ix])

[-0.038194 -0.24487   0.72812  -0.39961   0.083172  0.043953 -0.39141
  0.3344   -0.57545   0.087459  0.28787  -0.06731   0.30906  -0.26384
 -0.13231  -0.20757   0.33395  -0.33848  -0.31743  -0.48336   0.1464
 -0.37304   0.34577   0.052041  0.44946  -0.46971   0.02628  -0.54155
 -0.15518  -0.14107  -0.039722  0.28277   0.14393   0.23464  -0.31021
  0.086173  0.20397   0.52624   0.17164  -0.082378 -0.71787  -0.41531
  0.20335  -0.12763   0.41367   0.55187   0.57908  -0.33477  -0.36559
 -0.54857  -0.062892  0.26584   0.30205   0.99775  -0.80481  -3.0243
  0.01254  -0.36942   2.2167    0.72201  -0.24978   0.92136   0.034514
  0.46745   1.1079   -0.19358  -0.074575  0.23353  -0.052062 -0.22044
  0.057162 -0.15806  -0.30798  -0.41625   0.37972   0.15006  -0.53212
 -0.2055   -1.2526    0.071624  0.70565   0.49744  -0.42063   0.26148
 -1.538    -0.30223  -0.073438 -0.28312   0.37104  -0.25217   0.016215
 -0.017099 -0.38984   0.87424  -0.72569  -0.51058  -0.52028  -0.1459
  0.8278    0.27062 ]

In [60]:
dict_100 = bpemb_en_100.emb.index_to_key

In [61]:
dict_50 = bpemb_en_50.emb.index_to_key

In [62]:
dict_100 == dict_50

True

In [7]:
from flair.embeddings import WordEmbeddings, TokenEmbeddings
from flair.data import Token, Sentence
# init embedding
glove_embedding = WordEmbeddings('en-extvec')
#glove_embedding = TokenEmbeddings('en-twitter')


sentence = Sentence(a)
print(len(sentence))
glove_embedding.embed(sentence, embedding_length=100)
print(sentence.embedding)

mean_emb = 0
for token in sentence:
    print(token)
    print(token.embedding.shape)
    
print(mean_emb)

3


TypeError: embed() got an unexpected keyword argument 'embedding_length'

In [42]:
print(word)

Sentence[3]: "<user>"


In [63]:
sentence = Sentence(w2i[1])

# embed a sentence using glove.
glove_embedding.embed(sentence)

print(sentence.embedding)
# now check out the embedded tokens.
for token in sentence:
    print(token)
    print(token.embedding)

tensor([], device='cuda:0')
Token[0]: "."
tensor([ 0.1821, -0.0485,  0.2397,  0.3210, -0.2700,  0.7043, -0.2126,  0.2350,
         0.0901,  0.8214,  0.3784, -0.5638, -2.4447,  0.1683,  0.2468,  0.2865,
         0.0623,  0.0675, -0.5846, -0.4541, -0.2216,  0.1742, -0.3556,  0.1449,
         0.4909, -1.7426, -0.5431, -0.5194,  0.9480, -0.4174, -0.5524, -0.0574,
        -0.5266,  0.6298,  0.0973,  0.2064,  0.4626,  0.0895,  0.0160, -0.5385,
        -1.2043,  0.0803, -0.6535,  0.0446,  0.7953,  0.0445,  0.5337,  0.2744,
        -0.3246, -0.0537, -0.7930,  0.1100,  0.3976, -0.0442,  0.2170,  0.2798,
        -0.2577,  0.2508,  0.3971,  0.3232,  0.1024, -0.0305,  0.3411,  0.1797,
         0.4444,  0.0549,  0.2246, -0.8084, -0.1105,  0.4237,  0.6109,  0.5502,
         0.2196, -0.3029,  0.1454, -0.4670, -0.2394,  0.0351, -0.5093, -0.1239,
         0.2453,  0.1476, -0.3031, -0.5305,  0.8063,  0.3457, -0.2454,  0.7148,
        -0.1599,  0.4013, -0.1085, -0.6143, -0.0036,  0.0377, -0.3305, -0.0944

In [12]:
from flair.embeddings import FlairEmbeddings, TokenEmbeddings
from flair.data import Token, Sentence

# init embedding
flair_embedding_forward = FlairEmbeddings('it-forward')

# create a sentence
sentence = Sentence('The grass is green .')

# embed words in sentence
flair_embedding_forward.embed(sentence)

2024-10-08 09:41:46,812 https://flair.informatik.hu-berlin.de/resources/embeddings/flair/lm-it-opus-large-forward-v0.1.pt not found in cache, downloading to /tmp/tmp5suekufs


100%|██████████| 151M/151M [01:47<00:00, 1.47MB/s] 

2024-10-08 09:43:34,248 copying /tmp/tmp5suekufs to cache at /root/.flair/embeddings/lm-it-opus-large-forward-v0.1.pt
2024-10-08 09:43:34,409 removing temp file /tmp/tmp5suekufs


[Sentence[5]: "The grass is green ."]

In [13]:
for token in sentence:
    print(token.embedding.shape)

torch.Size([2048])
torch.Size([2048])
torch.Size([2048])
torch.Size([2048])
torch.Size([2048])


In [15]:
!git clone https://github.com/facebookresearch/fastText.git

Cloning into 'fastText'...
remote: Enumerating objects: 3998, done.
remote: Counting objects: 100% (1026/1026), done.
remote: Compressing objects: 100% (195/195), done.
remote: Total 3998 (delta 890), reused 859 (delta 826), pack-reused 2972 (from 1)
Receiving objects: 100% (3998/3998), 8.30 MiB | 10.05 MiB/s, done.
Resolving deltas: 100% (2528/2528), done.
Checking connectivity... done.


In [16]:
%cd fastText

/home/mounted/GW-Solvers/notebooks/fastText


In [24]:
!pip install .  --no-binary :all:

Processing /home/mounted/GW-Solvers/notebooks/fastText
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
    Preparing wheel metadata ... done
  ERROR: Command errored out with exit status 1:
   command: /opt/anaconda3/bin/python /opt/anaconda3/lib/python3.8/site-packages/pip/_vendor/pep517/_in_process.py build_wheel /tmp/tmpfbe40jpk
       cwd: /tmp/pip-req-build-a4_w6rvu
  Complete output (44 lines):
  running bdist_wheel
  running build
  running build_py
  creating build/lib.linux-x86_64-cpython-38/fasttext
  copying python/fasttext_module/fasttext/__init__.py -> build/lib.linux-x86_64-cpython-38/fasttext
  copying python/fasttext_module/fasttext/FastText.py -> build/lib.linux-x86_64-cpython-38/fasttext
  creating build/lib.linux-x86_64-cpython-38/fasttext/util
  copying python/fasttext_module/fasttext/util/util.py -> build/lib.linux-x86_64-cpython-38/fasttext/util
  copying python/fasttext_module/fasttext/util/__init__.py -> build/lib.linux-x8

In [3]:
import fasttext
import fasttext.util

#fasttext.util.download_model('en', if_exists='ignore')

ft = fasttext.load_model('cc.en.300.bin')
print(ft.get_dimension())

fasttext.util.reduce_model(ft, 100)
print(ft.get_dimension())

300
100


In [15]:
dim_red = 'pca-25'
i = dim_red.find('-')
print(dim_ref[i+1:])

25


In [10]:
import numpy as np
from sklearn import random_projection
X = np.random.rand(100, 300)
print(X.shape)
transformer = random_projection.GaussianRandomProjection(50)
X_new = transformer.fit_transform(X)
X_new.shape

(100, 300)


(100, 50)

In [1]:
import sys
sys.path.append('../')

In [4]:
from src import dataset_loaders_new
import src
SOURCE_LANG = 'en'
TARGET_LANG = 'es'

source_path = f'../datasets/muse/embeddings/wiki.multi.{SOURCE_LANG}.vec.txt'
target_path = f'../datasets/muse/embeddings/wiki.multi.{TARGET_LANG}.vec.txt'
dict_path = f'../datasets/muse/dictionaries/{SOURCE_LANG}-{TARGET_LANG}.txt'

source_orig = dataset_loaders_new.embeddings(source_path, dataset='muse2')
target_orig = dataset_loaders_new.embeddings(target_path, dataset='muse2')

In [8]:
vocab, src2trg_dict = src.utils.get_dict(dict_path, source_orig, target_orig)

valid_keys = []
for k in src2trg_dict.keys():
    if src2trg_dict[k] != set():
        valid_keys.append(k)

In [11]:
print(len(valid_keys))

93084


In [14]:
print(min(src2trg_dict[7]))

7
